In [1]:
import pandas as pd
import math

In [2]:
sheet_lst = ['sales', 'total_store_list', 'treatment_control_pairs', 'step_1']
sheet_dic = {}

for sheet in sheet_lst:
    sheet_dic[sheet] = pd.read_csv(f'materials/3-17_ab-test-walkthrough/sales-raw-{sheet}.csv')

In [3]:
tmp_df = sheet_dic['step_1']
tmp_df = tmp_df[tmp_df['Period']=='Comp'].groupby(['Store'], as_index=False)['Sum_Gross_Sales'].mean()
tmp_df = pd.DataFrame(tmp_df)
tmp_df.columns = ['Store', 'Avg Gross Sales for Entire Comp Period']

step1_df = pd.merge(sheet_dic['sales'], tmp_df, how='left', on='Store')
del tmp_df

step1_df['Gross Sales Growth'] = step1_df.apply(lambda row: round((row['Sum_Gross_Sales']-row['Avg Gross Sales for Entire Comp Period'])/row['Avg Gross Sales for Entire Comp Period'], 3)*100, axis=1)
step1_df.head(20)

,Region,Store,Week,Week_Start,Week_End,Period,Sum_Gross_Sales,Avg Gross Sales for Entire Comp Period,Gross Sales Growth
0,Midwest,store_012,49,2014/1/9,2014/1/15,Comp,1682.54,1796.796,-6.4
1,Midwest,store_012,50,2014/1/16,2014/1/22,Comp,2387.10,1796.796,32.9
2,Midwest,store_012,51,2014/1/23,2014/1/29,Comp,1847.11,1796.796,2.8
3,Midwest,store_012,52,2014/1/30,2014/2/5,Comp,1192.87,1796.796,-33.6
4,Midwest,store_012,53,2014/2/6,2014/2/12,Comp,2996.67,1796.796,66.8
5,Midwest,store_012,54,2014/2/13,2014/2/19,Comp,1800.50,1796.796,0.2
6,Midwest,store_012,55,2014/2/20,2014/2/26,Comp,1087.62,1796.796,-39.5
7,Midwest,store_012,56,2014/2/27,2014/3/5,Comp,1655.80,1796.796,-7.8
8,Midwest,store_012,57,2014/3/6,2014/3/12,Comp,2315.13,1796.796,28.8
9,Midwest,store_012,58,2014/3/13,2014/3/19,Comp,1002.62,1796.796,-44.2


In [4]:
tmp_df = step1_df.groupby(['Store', 'Period'], as_index=False)['Gross Sales Growth'].mean()
step2_df = tmp_df.pivot(index='Store', columns='Period')['Gross Sales Growth'].reset_index()

del tmp_df
step2_df.head(20)

Period,Store,Comp,Test
0,store_012,0.000000e+00,9.880000
1,store_015,-7.105427e-16,-54.290000
2,store_024,-1.000000e-02,20.010000
3,store_027,-1.111111e-02,-21.162500
4,store_029,1.000000e-02,-18.770000
5,store_033,-2.000000e-02,-50.390000
6,store_150,-7.105427e-16,-15.360000
7,store_251,0.000000e+00,-3.222222
8,store_299,-1.000000e-02,-8.080000
9,store_323,7.105427e-16,-11.570000


In [5]:
def pair(store, treatment_pairing, control):
    if control == 'Control':
        return f'{store} <-> {treatment_pairing}'
    else:
        return store

tmp_df = sheet_dic['treatment_control_pairs'][['Controls', 'Treatments', 'Region']]
tmp_df.columns = ['Store', 'Treatment-Pairing', 'Region']

tmp_df = pd.merge(sheet_dic['total_store_list'], tmp_df, how='left', on=['Store', 'Region'])
tmp_df['Control-Treatment Pairing'] = tmp_df.apply(lambda row: pair(row['Store'], row['Treatment-Pairing'], row['Type']), axis=1)

step3_df = tmp_df[['Store', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing']]
#step3_df.columns = ['store_no', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing']

del tmp_df
step3_df.head(20)

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing
0,store_015,NaN,Midwest,Treatment,store_015
1,store_024,NaN,Midwest,Treatment,store_024
2,store_027,NaN,Midwest,Treatment,store_027
3,store_695,NaN,Midwest,Treatment,store_695
4,store_696,NaN,Midwest,Treatment,store_696
5,store_707,NaN,Midwest,Treatment,store_707
6,store_719,NaN,Midwest,Treatment,store_719
7,store_721,NaN,Midwest,Treatment,store_721
8,store_724,NaN,Midwest,Treatment,store_724
9,store_727,NaN,Midwest,Treatment,store_727


In [11]:
step4_df = pd.merge(step3_df, step2_df, how='left')
step4_df.columns = ['Store', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing', 'Avg Growth Comparative Period', 'Avg Growth Test Period']

step4_df.to_csv('step4_df.csv')

step4_df

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing,Avg Growth Comparative Period,Avg Growth Test Period
0,store_015,NaN,Midwest,Treatment,store_015,-7.105427e-16,-54.2900
1,store_024,NaN,Midwest,Treatment,store_024,-1.000000e-02,20.0100
2,store_027,NaN,Midwest,Treatment,store_027,-1.111111e-02,-21.1625
3,store_695,NaN,Midwest,Treatment,store_695,2.000000e-02,-40.6800
4,store_696,NaN,Midwest,Treatment,store_696,8.437695e-16,-37.4500
...,...,...,...,...,...,...,...
78,store_636,store_769,Midwest,Control,store_636 <-> store_769,1.000000e-02,-26.2800
79,store_594,store_596,West,Control,store_594 <-> store_596,1.421085e-15,4.2200
80,store_526,store_596,West,Control,store_526 <-> store_596,1.000000e-02,-33.1100
81,store_478,store_601,West,Control,store_478 <-> store_601,-1.776357e-15,-1.4000


In [10]:
step5_df = step4_df.copy()
step5_df['Growth_Difference'] = step5_df.apply(lambda row: row['Avg Growth Test Period'] - row['Avg Growth Comparative Period'], axis=1)
step5_df

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing,Avg Growth Comparative Period,Avg Growth Test Period,Growth_Difference
0,store_015,NaN,Midwest,Treatment,store_015,-7.105427e-16,-54.2900,-54.290000
1,store_024,NaN,Midwest,Treatment,store_024,-1.000000e-02,20.0100,20.020000
2,store_027,NaN,Midwest,Treatment,store_027,-1.111111e-02,-21.1625,-21.151389
3,store_695,NaN,Midwest,Treatment,store_695,2.000000e-02,-40.6800,-40.700000
4,store_696,NaN,Midwest,Treatment,store_696,8.437695e-16,-37.4500,-37.450000
...,...,...,...,...,...,...,...,...
78,store_636,store_769,Midwest,Control,store_636 <-> store_769,1.000000e-02,-26.2800,-26.290000
79,store_594,store_596,West,Control,store_594 <-> store_596,1.421085e-15,4.2200,4.220000
80,store_526,store_596,West,Control,store_526 <-> store_596,1.000000e-02,-33.1100,-33.120000
81,store_478,store_601,West,Control,store_478 <-> store_601,-1.776357e-15,-1.4000,-1.400000
